# MP2 Part 3.2: Interpreting the Longest Gap (lwang111)

For each project, this notebook records why the project went quiet during its longest gap, whether it recovered, why, and who drove the recovery. It also checks each project's current status on GitHub and labels the themes of its 10 most recent commits. Everything is written to new columns in `lwang111_project_stats.csv`.

Evidence comes from the commit messages in `lwang111_project_gap_commits.csv` first, then from GitHub issues, pull requests, and README history around each gap. The longer write-ups are in `reflections/lwang111_reflection_<WOCProjectID>.md`.

Run `lwang111_gap_analysis.ipynb` first, because this notebook builds on the columns it writes.

In [1]:
import os
import time

import pandas as pd
import requests

NETID = "lwang111"
RECENT_COMMIT_COUNT = 10

# A project is Active if it has a GitHub commit in the 6 months before 2025-09-30.
ACTIVE_CUTOFF_DATE = "2025-04-01"
RECENT_COMMITS_CACHE = f"{NETID}_recent_commits.csv"

## Load commits, gap boundaries, and GitHub URLs

In [2]:
commits = pd.read_csv(f"{NETID}_project_summary.csv", sep=";")
commit_datetimes = pd.to_datetime(commits["time"], unit="s")
commits["Month"] = commit_datetimes.dt.strftime("%Y-%m")
commits["author_name"] = commits["author"].str.split("<").str[0].str.strip()

project_stats = pd.read_csv(f"{NETID}_project_stats.csv", sep=";")

assigned_projects = pd.read_csv("net2prj.csv", header=None, names=["netid", "project", "github_url"])
assigned_projects = assigned_projects[assigned_projects["netid"] == NETID]
github_repos = assigned_projects.set_index("project")["github_url"].str.replace("https://github.com/", "")
github_repos

project
libatoms_quip                                                    libAtoms/QUIP
ldbc_sigmod2014-pc                                          ldbc/sigmod2014-pc
wurmlab_sequenceserver                                  wurmlab/sequenceserver
lanl_elements                                                    lanl/ELEMENTS
homalg-project_toricvarieties_project    homalg-project/ToricVarieties_project
ph-rast_bmgarch                                                ph-rast/bmgarch
nanocomp_meep                                                    NanoComp/meep
wiris_moodle-filter_wiris                            wiris/moodle-filter_wiris
asgr_profound                                                    asgr/ProFound
opticspy_lightpipes                                        opticspy/lightpipes
Name: github_url, dtype: str

## Who committed after the gap

To answer WhoRecovered, every author who committed after the gap is counted and checked against everyone who committed at any point before it. Some people show up under two names (e.g. `stevenj` and `Steven G. Johnson` in meep, `Chris Hogan` and `Christopher Hogan`), so the table is read by hand rather than trusted blindly.

In [3]:
# Count each author's commits after the gap and flag whether they had committed before it.
def summarize_authors_after_gap(project_commits, gap_start_month, gap_end_month):
    is_before_gap = project_commits["Month"] < gap_start_month
    is_after_gap = project_commits["Month"] > gap_end_month
    authors_before_gap = set(project_commits.loc[is_before_gap, "author_name"])

    author_counts = project_commits.loc[is_after_gap, "author_name"].value_counts()
    author_summary = author_counts.rename("commits_after_gap").to_frame()
    author_summary["committed_before_gap"] = author_summary.index.isin(authors_before_gap)

    return author_summary


for _, row in project_stats.iterrows():
    project = row["project"]
    project_commits = commits[commits["project_wocid"] == project]
    author_summary = summarize_authors_after_gap(
        project_commits,
        row["LongestGapStart (YYYY-MM)"],
        row["LongestGapEnd (YYYY-MM)"],
    )

    print(f"===== {project}")
    print(author_summary.head(8).to_string())
    print()

===== asgr_profound
                commits_after_gap  committed_before_gap
author_name                                            
Aaron Robotham                 81                  True
Joel G                          1                 False
Rodrigo Tobar                   1                  True

===== homalg-project_toricvarieties_project
             commits_after_gap  committed_before_gap
author_name                                         
Martin Bies                  4                  True
Max Horn                     3                  True

===== lanl_elements
                  commits_after_gap  committed_before_gap
author_name                                              
Jacob Moore                      76                 False
Adrian-Diaz                       8                  True
Nathaniel Morgan                  4                  True
Daniel Dunning                    2                  True

===== ldbc_sigmod2014-pc
               commits_after_gap  committed_befo

## Current status and most recent commits on GitHub

WoC data stops before the latest GitHub activity, so the 10 most recent commits are pulled from the GitHub API. Results are cached in `lwang111_recent_commits.csv` so re-running the notebook does not use up the 60-requests-per-hour unauthenticated limit.

In [4]:
# Fetch the most recent commits on a repo's default branch from the GitHub API.
def fetch_recent_github_commits(repo):
    url = f"https://api.github.com/repos/{repo}/commits"
    params = {"per_page": RECENT_COMMIT_COUNT}
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    rows = []
    for item in response.json():
        commit_info = item["commit"]
        first_message_line = commit_info["message"].strip().splitlines()[0]
        rows.append({
            "date": commit_info["author"]["date"][:10],
            "author": commit_info["author"]["name"],
            "message": first_message_line,
        })

    return rows


if os.path.exists(RECENT_COMMITS_CACHE):
    recent_commits = pd.read_csv(RECENT_COMMITS_CACHE, sep=";")
else:
    recent_rows = []
    for project, repo in github_repos.items():
        for row in fetch_recent_github_commits(repo):
            recent_rows.append({"project": project, **row})
        time.sleep(1)

    recent_commits = pd.DataFrame(recent_rows)
    recent_commits.to_csv(RECENT_COMMITS_CACHE, sep=";", index=False)

# GitHub lists commits in branch order, not strictly by date, so sort before reading.
recent_commits = recent_commits.sort_values(["project", "date"], ascending=[True, False])

for project, project_recent in recent_commits.groupby("project"):
    print(f"===== {project}")
    for _, row in project_recent.iterrows():
        print(f"{row['date']}  {row['author'][:20]:<20} {row['message'][:100]}")
    print()

===== asgr_profound
2026-09-26  Aaron Robotham       Merge branch 'master' of https://github.com/asgr/ProFound
2026-09-26  Aaron Robotham       Lots of speed-ups and large amount of tests offering good coverage.
2026-07-07  Aaron Robotham       Doc fix
2026-03-11  Aaron Robotham       Regular Sunday runs added
2026-03-04  Aaron Robotham       Version bump
2026-03-04  Aaron Robotham       Make sure it uses floating point arithmetic.
2026-02-26  Aaron Robotham       Simplify installation instructions in README
2026-02-26  Aaron Robotham       Merge pull request #31 from asgr/copilot/fix-broken-remotes-description-file
2026-02-26  copilot-swe-agent[bo Fix broken Remotes in DESCRIPTION: add commas between entries
2026-02-26  copilot-swe-agent[bo Initial plan

===== homalg-project_toricvarieties_project
2025-01-13  Martin Bies          Update Readme.md
2025-01-13  Martin Bies          Update Readme.md
2024-12-20  Max Horn             Avoid using InstallValue on non-plain objects
2022-12-28 

In [5]:
has_recent_commit = project_stats["lastGHCommitDate"] >= ACTIVE_CUTOFF_DATE
project_stats["Currentstatus"] = has_recent_commit.map({True: "Active", False: "Inactive"})
project_stats[["project", "lastGHCommitDate", "Currentstatus"]]

,project,lastGHCommitDate,Currentstatus
0,asgr_profound,2026-09-26,Active
1,homalg-project_toricvarieties_project,2025-01-14,Inactive
2,lanl_elements,2026-09-22,Active
3,ldbc_sigmod2014-pc,2025-05-30,Active
4,libatoms_quip,2026-05-12,Active
5,nanocomp_meep,2026-09-27,Active
6,opticspy_lightpipes,2026-02-26,Active
7,ph-rast_bmgarch,2026-07-07,Active
8,wiris_moodle-filter_wiris,2026-05-29,Active
9,wurmlab_sequenceserver,2026-05-11,Active


## Recent themes

Same categories as Part 3.1. Ties are broken toward the theme that better describes what the project is spending its effort on right now.

| Project | Recent commits | RecentThemes |
|---|---|---|
| asgr_profound | README install fix, doc fix, broken `Remotes` in DESCRIPTION (PR #31, partly by Copilot), version bump, floating point fix, Sunday CI runs | Documentation updates:Dependency updates |
| homalg-project_toricvarieties_project | README edits to retire the package, then the Dec 2022 refactors (old methods removed, `.gitignore`) | Other:Documentation updates |
| lanl_elements | boundary surface sets for Fierro, notched circle, limiter (new features), mesh reading and tensor rank fixes | Feature development:Bug fixes |
| ldbc_sigmod2014-pc | "Fix missing import", DOI, LICENSE/NOTICE, README, then CI tweaks from 2020 (sha1 sum, zstd, sudo) | Other:Documentation updates |
| libatoms_quip | ScaLAPACK linking, segfault line size, missing CLI executables, copy/paste fix; CMake support, `qr` build option, binary sparseX reading | Bug fixes:Feature development |
| nanocomp_meep | Windows native build, Apple silicon CI, mingw CI, removing nlopt from CI, test tolerances, Makefile fixes, buffer overrun fix | Other:Feature development |
| opticspy_lightpipes | Michelson and Poisson spot examples, install docs, broken image link, "fixed several bugs due to numpy expirations", 7.1.1 | Documentation updates:Bug fixes |
| ph-rast_bmgarch | GitHub Actions YAML fixes, unit tests, CRAN notes, `R CMD check` fixes, v2.1.0 | Other:Release |
| wiris_moodle-filter_wiris | Moodle 5.2/5.3 beta tests, CODEOWNERS, class alias and corrupted character fixes, publish 8.11.0/8.11.1 | Other:Bug fixes |
| wurmlab_sequenceserver | Ruby 3.3 security update, hide backtrace in prod, code scanning alert fix; `num_threads` caching and Docker build fixes | Security patches:Bug fixes |

In [6]:
RECENT_THEMES = {
    "asgr_profound": "Documentation updates:Dependency updates",
    "homalg-project_toricvarieties_project": "Other:Documentation updates",
    "lanl_elements": "Feature development:Bug fixes",
    "ldbc_sigmod2014-pc": "Other:Documentation updates",
    "libatoms_quip": "Bug fixes:Feature development",
    "nanocomp_meep": "Other:Feature development",
    "opticspy_lightpipes": "Documentation updates:Bug fixes",
    "ph-rast_bmgarch": "Other:Release",
    "wiris_moodle-filter_wiris": "Other:Bug fixes",
    "wurmlab_sequenceserver": "Security patches:Bug fixes",
}

## Gap interpretation

Commit messages are the main evidence. GitHub issues, PRs, and README history fill in what the messages leave out.

| Project | Why it went quiet | Recovered? | Why it came back | Who brought it back |
|---|---|---|---|---|
| asgr_profound | One-person project (Aaron Robotham made 81 of 83 post-gap commits). The last pre-gap commit finished the Rfits_pointer feature and no issues were opened during the gap, so there was nothing pulling him back. | Yes | His own research needs: new `segim_orig` photometry modes, "Will need this version for the Workshop", and a "corner case for WAVES". | Same maintainer; Joel G's single "upgrade cicd" PR (#26) is the only outside help. |
| homalg-project_toricvarieties_project | Replaced by OSCAR. In PR #206 Martin Bies says all its functionality "is now available through OSCAR", and Mohamed Barakat calls it "incompatible with our current philosophy". Issue #205 (Normaliz segfault, July 2023) was never fixed; tests were disabled instead. | No | The 7 commits only wind it down: Max Horn (GAP) fixed compatibility and asked if it was still maintained, which led to withdrawing it from GAP. | Same people: Martin Bies and GAP maintainer Max Horn, who had committed before the gap. |
| lanl_elements | Pre-gap work was mostly MATAR submodule bumps and merges, so ELEMENTS was mainly being kept in sync for other LANL codes. Work likely went to those codes (MATAR, Fierro). | Yes | "updated matar version (for updated Kokkos)" restarted it. Later a v2 refactor (#39) and new boundary surface sets "to be used in fierro". | Mostly a newcomer: Jacob Moore made 76 of 90 post-gap commits. |
| ldbc_sigmod2014-pc | A contest solution / research artifact that was finished. The last issues (#43, #44, early 2021) are open ideas, and the last pre-gap commit just adds a DOI to archive it. | No | One "Fix missing import" commit in May 2025 is a small cleanup, not a restart. | Original contributor Márton Elekes. |
| libatoms_quip | Maintainers stopped committing while users kept filing issues (#669 has 30 comments, #673 has 23, #675-#682 mostly open). Python packaging was breaking (#674 Python 3.12, #687 "CI tests broken"). | Yes | Build modernization: renaming F95 to F90, then meson builds, NumPy 2.x support, and fixed wheels to get installs working again. | Same core maintainers (James Kermode, Albert Bartok-Partay), joined by new contributors (green-br, smjanke) and returning contributor Noam Bernstein. |
| nanocomp_meep | Steven Johnson had just shipped 1.2.1 and was the only committer, so the code sat stable. The history was also being moved from SVN to git. | Yes | "git conversion" and "add mailmap file" in March 2014 moved it to GitHub, and outside users started filing issues within weeks (#1-#7). | Same maintainer at first (`stevenj` = Steven G. Johnson); long-term growth came from new people (Ardavan Oskooi, Chris Hogan, Homer Reid). |
| opticspy_lightpipes | Single developer (Fred van Goor) finished a docs-and-release push ("add newest version", README edits) and stopped. | Yes | A move to the opticspy repo with a partner: issues "Init repository", "Invite more members", and "build linux wheels" in Jan 2017, plus FFTW3 builds. | Fred van Goor plus a new contributor, guyskk, who handled packaging and CI. |
| ph-rast_bmgarch | Pre-gap commits are all CRAN submission prep, so the package was released and considered done. Issue #19 sat unanswered for 6 months; the maintainer said "I did not get any emails from this repo". | Yes | Outside pressure: Andrew Johnson's PR #20 "Update deprecated syntax for future rstan compatibility", which then led to CMD check and CI fixes. | Same people: Philippe Rast and Andrew Johnson, who also contributed before the gap (PR #17). |
| wiris_moodle-filter_wiris | N/A active Project. Longest pause was 2 months (Aug-Sep 2022) and the project has no 3+ month gaps. | Yes | Normal work resumed: Moodle 4.1 tests and the 8.0.0 release. | Same company team (carla-at-wiris, icaparros-at-wiris, Xavier Ripoll). |
| wurmlab_sequenceserver | Short 4-month review backlog. PR #823 (Dec 2024) waited until June 2025, Dependabot PR #822 is still open, and issues #824-#826 got no replies during the gap. | Yes | A newcomer fixed issue #826 (PR #828), and Tadas Tamosauskas merged the backlog on 2025-06-09. | Mostly the existing team (Tadas Tamosauskas, Joko Triono) plus newcomer Good-soup1. |

In [7]:
HYPOTHESIZED_GAP_REASON = {
    "asgr_profound": "Single-maintainer project paused after finishing a feature. Aaron Robotham made 81 of 83 post-gap commits; last pre-gap commit completed Rfits_pointer loading and no issues were opened during the gap.",
    "homalg-project_toricvarieties_project": "Replaced by OSCAR. PR #206: functionality 'is now available through OSCAR'; co-owner called it 'incompatible with our current philosophy'. Issue #205 (Normaliz segfault, Jul 2023) left unfixed and tests disabled.",
    "lanl_elements": "Work shifted to dependent LANL codes. Pre-gap commits were mostly MATAR submodule bumps and merges, so the library was only being kept in sync.",
    "ldbc_sigmod2014-pc": "Research/contest artifact considered finished. Last pre-gap commit adds a DOI to archive it; last issues (#43, #44 from 2021) are open ideas never picked up.",
    "libatoms_quip": "Maintainers busy while user issues piled up (#669 30 comments, #673 23 comments, #675-#682 mostly open) and Python packaging broke (#674 Python 3.12, #687 'CI tests broken').",
    "nanocomp_meep": "Single maintainer (stevenj) paused after preparing the 1.2.1 release; code was stable and the history was being converted from SVN to git.",
    "opticspy_lightpipes": "Single developer finished a docs/release push ('add newest version', README edits) and stopped.",
    "ph-rast_bmgarch": "Package released to CRAN and considered done (pre-gap commits are all CRAN submission prep). Issue #19 went unanswered 6 months because the maintainer 'did not get any emails from this repo'.",
    "wiris_moodle-filter_wiris": "N/A active Project",
    "wurmlab_sequenceserver": "Short review backlog: PR #823 opened Dec 2024 was not merged until Jun 2025, Dependabot PR #822 still open, issues #824-#826 had no replies during the gap.",
}

HAS_RECOVERED = {
    "asgr_profound": "Yes",
    "homalg-project_toricvarieties_project": "No",
    "lanl_elements": "Yes",
    "ldbc_sigmod2014-pc": "No",
    "libatoms_quip": "Yes",
    "nanocomp_meep": "Yes",
    "opticspy_lightpipes": "Yes",
    "ph-rast_bmgarch": "Yes",
    "wiris_moodle-filter_wiris": "Yes",
    "wurmlab_sequenceserver": "Yes",
}

WHY_RECOVERED = {
    "asgr_profound": "Maintainer's own research needs: new segim_orig photometry modes, 'Will need this version for the Workshop', and a WAVES survey corner case fix.",
    "homalg-project_toricvarieties_project": "N/A. The 7 post-gap commits only wind it down: a GAP compatibility fix by Max Horn led to the decision to withdraw it from GAP and update the README.",
    "lanl_elements": "MATAR/Kokkos update ('updated matar version (for updated Kokkos)'), then a v2 refactor (#39, Feb 2026) and boundary surface sets 'to be used in fierro'.",
    "ldbc_sigmod2014-pc": "N/A. Only one post-gap commit ('Fix missing import', May 2025), a small cleanup.",
    "libatoms_quip": "Build modernization to fix broken installs: F95 to F90 rename (Apr 2025), then meson build, NumPy 2.x support, and fixed quippy wheels.",
    "nanocomp_meep": "Moved to git/GitHub ('git conversion', 'add mailmap file', Mar 2014); outside users opened issues #1-#7 within weeks, creating demand.",
    "opticspy_lightpipes": "Moved to the opticspy repo with a new partner (issues 'Init repository', 'Invite more members', 'build linux wheels', Jan 2017) and FFTW3-accelerated builds.",
    "ph-rast_bmgarch": "External PR #20 'Update deprecated syntax for future rstan compatibility' (Sep 2023) forced an update, followed by CMD check and CI fixes.",
    "wiris_moodle-filter_wiris": "Normal work resumed after a 2-month pause: Moodle 4.1 tests and the 8.0.0 release.",
    "wurmlab_sequenceserver": "Newcomer PR #828 fixed issue #826, then the maintainer merged the backlog (#823, #828, #829, #830) on 2025-06-09.",
}

WHO_RECOVERED = {
    "asgr_profound": "Same maintainer (Aaron Robotham); only one outside commit (Joel G, 'upgrade cicd').",
    "homalg-project_toricvarieties_project": "Same people: Martin Bies and GAP maintainer Max Horn, who had also committed before the gap; no new developers.",
    "lanl_elements": "Mostly new: Jacob Moore made 76 of 90 post-gap commits; existing team (Adrian-Diaz, Nathaniel Morgan, Daniel Dunning) contributed the rest.",
    "ldbc_sigmod2014-pc": "Same person (Márton Elekes) for the single post-gap commit.",
    "libatoms_quip": "Same core maintainers (James Kermode 48, Albert Bartok-Partay 31) plus new contributors (green-br, smjanke) and returning contributor Noam Bernstein.",
    "nanocomp_meep": "Same maintainer restarted it (stevenj = Steven G. Johnson); new people (Ardavan Oskooi, Chris Hogan, Homer Reid) drove later growth.",
    "opticspy_lightpipes": "Same developer (Fred van Goor) plus a new contributor (guyskk) who set up wheels and CI.",
    "ph-rast_bmgarch": "Same people: Philippe Rast (18 of 19) and Andrew Johnson, who also contributed before the gap.",
    "wiris_moodle-filter_wiris": "Same company team (carla-at-wiris, icaparros-at-wiris, Xavier Ripoll).",
    "wurmlab_sequenceserver": "Mostly existing team (Tadas Tamosauskas, Joko Triono) plus newcomer Good-soup1.",
}

## Notes

A 2-4 sentence summary of each project's reflection file.

In [8]:
NOTES = {
    "asgr_profound": "ProFound is basically one astronomer's tool, so its activity follows his research schedule. The 6-month gap in 2024 was easy to read: he finished a feature, had no open issues, and came back when he needed new photometry options for a workshop and the WAVES survey. It is still active in 2026.",
    "homalg-project_toricvarieties_project": "The 23-month gap was the start of the end, not a pause. PR #206 says the package was replaced by OSCAR and it was withdrawn from GAP in Jan 2025. The last 7 commits only wind it down, and it is now Inactive.",
    "lanl_elements": "ELEMENTS goes quiet whenever LANL's other codes do not need it, which explains its 3 gaps. The 6-month gap was moderately hard to read since the commits are mostly submodule bumps. It recovered through a Kokkos update and later a v2 refactor led by newcomer Jacob Moore.",
    "ldbc_sigmod2014-pc": "This is a finished research artifact, so its 33-month gap is really the project being done. Adding a DOI right before the gap was the clearest sign. One import fix in 2025 technically makes it Active under the 6-month rule, but it has not recovered.",
    "libatoms_quip": "QUIP has only one 3+ month gap in 15 years, so the 6-month pause stands out. Issues kept piling up while Python packaging broke, which made the cause fairly clear. The core maintainers came back to modernize the build and are active again with some new contributors.",
    "nanocomp_meep": "Meep went from a one-person project with 8 gaps to a busy multi-developer project. The 15-month gap was hard to read from commits alone, but the post-gap git conversion and new GitHub issues show the move to GitHub is what brought it back. It is very active today.",
    "opticspy_lightpipes": "LightPipes is a one-developer project with 9 gaps, so it moves in bursts. The 12-month gap in 2016 was easy to read: a docs/release push ended and nothing came in until a new partner helped move it to the opticspy repo. It is still updated occasionally.",
    "ph-rast_bmgarch": "bmgarch goes quiet after each CRAN release. The 20-month gap was easy to read because the maintainer admitted missing issue notifications, and an outside rstan compatibility PR brought it back. It is active again with a v2.1.0 release in 2026.",
    "wiris_moodle-filter_wiris": "This company-maintained plugin has no 3+ month gaps, so it counts as always active. Its longest pause was 2 months over the 2022 summer, and work resumed with the normal Moodle 4.1 and 8.0.0 release cycle.",
    "wurmlab_sequenceserver": "SequenceServer has a cyclical pattern of short pauses. Its 4-month gap was a review backlog where PRs and issues waited without replies. It recovered when a newcomer fixed an issue and a maintainer merged the queue in June 2025.",
}

## Add Part 3.2 columns to lwang111_project_stats.csv

In [9]:
# Re-running replaces earlier Part 3.2 columns instead of duplicating them.
part_3_2_columns = {
    "HypothesizedGapReason": HYPOTHESIZED_GAP_REASON,
    "HasRecovered": HAS_RECOVERED,
    "WhyRecovered": WHY_RECOVERED,
    "WhoRecovered": WHO_RECOVERED,
    "RecentThemes": RECENT_THEMES,
    "Notes": NOTES,
}
project_stats = project_stats.drop(columns=list(part_3_2_columns), errors="ignore")

for column, values_by_project in part_3_2_columns.items():
    project_stats[column] = project_stats["project"].map(values_by_project)

# Currentstatus belongs between WhoRecovered and RecentThemes in the final column order.
column_order = [column for column in project_stats.columns if column != "Currentstatus"]
recent_themes_position = column_order.index("RecentThemes")
column_order.insert(recent_themes_position, "Currentstatus")
project_stats = project_stats[column_order]

project_stats.to_csv(f"{NETID}_project_stats.csv", sep=";", index=False)
project_stats[["project", "HasRecovered", "Currentstatus", "RecentThemes"]]

,project,HasRecovered,Currentstatus,RecentThemes
0,asgr_profound,Yes,Active,Documentation updates:Dependency updates
1,homalg-project_toricvarieties_project,No,Inactive,Other:Documentation updates
2,lanl_elements,Yes,Active,Feature development:Bug fixes
3,ldbc_sigmod2014-pc,No,Active,Other:Documentation updates
4,libatoms_quip,Yes,Active,Bug fixes:Feature development
5,nanocomp_meep,Yes,Active,Other:Feature development
6,opticspy_lightpipes,Yes,Active,Documentation updates:Bug fixes
7,ph-rast_bmgarch,Yes,Active,Other:Release
8,wiris_moodle-filter_wiris,Yes,Active,Other:Bug fixes
9,wurmlab_sequenceserver,Yes,Active,Security patches:Bug fixes
